In [ ]:
# Benchmark and reproducibility pipeline for multimodal motion prediction

import os
import re
import glob
import random
import warnings
import numpy as np
import pandas as pd
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0, ResNet50
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet50_preprocess

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# =========================================================
# 1. SETTINGS
# =========================================================
MAIN_DIR = os.getcwd()
DATA_DIR = os.path.join(MAIN_DIR, "data")
FREQUENCY_FOLDERS = ["0p5Hz", "1p0Hz", "2p0Hz"]

IMG_SIZE = (224, 224)
BATCH_SIZE = 8
EPOCHS = 50
LEARNING_RATE = 1e-4
SEED = 4250
ANGULAR_LOSS_WEIGHT = 1.0
BA_LOSS_WEIGHT = 1.0

SPLIT_NAME = "train12_test3"
TRAIN_CYCLES = [1, 2]
TEST_CYCLES = [3]

BACKBONE_CONFIGS = [
    ("mobilenetv2_frozen", "MobileNetV2"),
    ("efficientnetb0_frozen", "EfficientNetB0"),
    ("resnet50_frozen", "ResNet50"),
]

MODALITY_ORDER = [
    "2D ML",
    "3D ML",
    "DIC",
    "PR",
    "3D ML + DIC",
    "3D ML + PR",
    "DIC + PR",
    "3D ML + DIC + PR",
]

EXPERIMENT_CONFIGS = [
    {"Modalities": "2D ML", "ML_Type": "2D", "ML_Path_Column": "ml2d_path", "Use_ML": True,  "Use_DIC": False, "Use_PR": False},
    {"Modalities": "3D ML", "ML_Type": "3D", "ML_Path_Column": "ml3d_path", "Use_ML": True,  "Use_DIC": False, "Use_PR": False},
    {"Modalities": "DIC", "ML_Type": "None", "ML_Path_Column": None, "Use_ML": False, "Use_DIC": True,  "Use_PR": False},
    {"Modalities": "PR", "ML_Type": "None", "ML_Path_Column": None, "Use_ML": False, "Use_DIC": False, "Use_PR": True},
    {"Modalities": "3D ML + DIC", "ML_Type": "3D", "ML_Path_Column": "ml3d_path", "Use_ML": True,  "Use_DIC": True,  "Use_PR": False},
    {"Modalities": "3D ML + PR", "ML_Type": "3D", "ML_Path_Column": "ml3d_path", "Use_ML": True,  "Use_DIC": False, "Use_PR": True},
    {"Modalities": "DIC + PR", "ML_Type": "None", "ML_Path_Column": None, "Use_ML": False, "Use_DIC": True,  "Use_PR": True},
    {"Modalities": "3D ML + DIC + PR", "ML_Type": "3D", "ML_Path_Column": "ml3d_path", "Use_ML": True,  "Use_DIC": True,  "Use_PR": True},
]

RESULTS_DIR = os.path.join(MAIN_DIR, "results")
MODEL_DIR = os.path.join(RESULTS_DIR, "saved_weights")
FIGURE_DIR = os.path.join(RESULTS_DIR, "manuscript_figures")
PREDICTION_DIR = os.path.join(RESULTS_DIR, "test_predictions")
HISTORY_DIR = os.path.join(RESULTS_DIR, "training_history")

for d in [RESULTS_DIR, MODEL_DIR, FIGURE_DIR, PREDICTION_DIR, HISTORY_DIR]:
    os.makedirs(d, exist_ok=True)

# =========================================================
# 2. REPRODUCIBILITY
# =========================================================
def set_all_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_all_seeds(SEED)
print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

# =========================================================
# 4. HELPERS
# =========================================================
def natural_key(text):
    return [int(c) if c.isdigit() else c.lower() for c in re.split(r"(\d+)", str(text))]

def safe_name(text):
    return re.sub(r"[^A-Za-z0-9]+", "_", str(text)).strip("_")

def experiment_name(backbone_key, modality_name):
    return f"{backbone_key}__{safe_name(modality_name)}"

def list_images(folder):
    exts = ["*.png", "*.jpg", "*.jpeg", "*.bmp", "*.tif", "*.tiff"]
    files = []
    for ext in exts:
        files.extend(glob.glob(os.path.join(folder, ext)))
    files = sorted(files, key=natural_key)
    if len(files) == 0:
        raise FileNotFoundError(f"No images found in: {folder}")
    return files

def find_existing_dir(parent, candidates, description):
    tested = []
    for cand in candidates:
        path = os.path.join(parent, cand)
        tested.append(path)
        if os.path.isdir(path):
            return path
    raise FileNotFoundError(
        f"Could not find {description}. Tested folders:\n" + "\n".join(tested)
    )

def find_data_file(folder):
    files = []
    files.extend(glob.glob(os.path.join(folder, "*.xlsx")))
    files.extend(glob.glob(os.path.join(folder, "*.xls")))
    files.extend(glob.glob(os.path.join(folder, "*.csv")))
    files = sorted(files, key=natural_key)
    if len(files) == 0:
        raise FileNotFoundError(f"No Excel/CSV file found in: {folder}")
    return files[0]

def read_table(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    if ext in [".xlsx", ".xls"]:
        return pd.read_excel(file_path)
    elif ext == ".csv":
        return pd.read_csv(file_path)
    else:
        raise ValueError(f"Unsupported file type: {file_path}")

def find_time_column(columns):
    candidates = [
        "time_s", "time", "Time", "Time_s", "time(sec)", "time_sec",
        "seconds", "sec", "time (s)", "Time (s)"
    ]
    lower_map = {str(c).strip().lower(): c for c in columns}

    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]

    for c in columns:
        if "time" in str(c).strip().lower():
            return c

    raise ValueError(f"No time column found. Available columns: {list(columns)}")

# =========================================================
# 6. DATAFRAME BUILDING
# =========================================================
def build_full_dataframe(base_dir, frequency_folders):
    rows = []

    for freq in frequency_folders:
        freq_dir = os.path.join(base_dir, freq)
        if not os.path.isdir(freq_dir):
            raise FileNotFoundError(f"Frequency folder not found: {freq_dir}")

        for cycle_num in [1, 2, 3]:
            ml2d_dir = find_existing_dir(
                freq_dir,
                [
                    f"2D_ML_cycle{cycle_num}",
                    f"ML_cycle{cycle_num}",
                    f"2DML_cycle{cycle_num}",
                ],
                description=f"2D ML folder for {freq} cycle {cycle_num}"
            )

            ml3d_dir = find_existing_dir(
                freq_dir,
                [
                    f"3D_ML_cycle{cycle_num}",
                    f"3DML_cycle{cycle_num}",
                    f"ML3D_cycle{cycle_num}",
                ],
                description=f"3D ML folder for {freq} cycle {cycle_num}"
            )

            dic_dir = find_existing_dir(
                freq_dir,
                [
                    f"UV_DIC_cycle{cycle_num}",
                    f"UV_DIC_cycle{cycle_num}{cycle_num}",
                    f"DIC_cycle{cycle_num}",
                ],
                description=f"DIC folder for {freq} cycle {cycle_num}"
            )

            pr_dir = find_existing_dir(
                freq_dir,
                [
                    f"pr_cycle{cycle_num}",
                    f"PR_cycle{cycle_num}",
                    f"Pr_cycle{cycle_num}",
                ],
                description=f"PR folder for {freq} cycle {cycle_num}"
            )

            ml2d_images = list_images(ml2d_dir)
            ml3d_images = list_images(ml3d_dir)
            dic_images = list_images(dic_dir)
            pr_file = find_data_file(pr_dir)
            df = read_table(pr_file)

            required_cols = ["Round avg BA", "angular rate", "resistance_var"]
            for col in required_cols:
                if col not in df.columns:
                    raise ValueError(f"Column '{col}' not found in {pr_file}")

            time_col = find_time_column(df.columns)

            n_2d = len(ml2d_images)
            n_3d = len(ml3d_images)
            n_dic = len(dic_images)
            n_tab = len(df)

            if not (n_2d == n_3d == n_dic == n_tab):
                raise ValueError(
                    f"Mismatch in {freq} cycle{cycle_num}: "
                    f"2D ML={n_2d}, 3D ML={n_3d}, DIC={n_dic}, Table={n_tab}"
                )

            for i in range(n_tab):
                rows.append({
                    "frequency_folder": freq,
                    "cycle_num": cycle_num,
                    "ml2d_path": ml2d_images[i],
                    "ml3d_path": ml3d_images[i],
                    "dic_path": dic_images[i],
                    "round_avg_ba": float(df.iloc[i]["Round avg BA"]),
                    "angular_rate": float(df.iloc[i]["angular rate"]),
                    "resistance_var": float(df.iloc[i]["resistance_var"]),
                    "time_s": float(df.iloc[i][time_col]),
                })

    return pd.DataFrame(rows)

# =========================================================
# 7. IMAGE PREPROCESSING
# =========================================================
def load_and_preprocess_image(path, img_size=IMG_SIZE, backbone_type="mobilenetv2_frozen"):
    img = Image.open(path).convert("RGB")
    img = img.resize(img_size)
    img = np.array(img).astype(np.float32)

    if backbone_type == "mobilenetv2_frozen":
        img = mobilenet_preprocess(img)
    elif backbone_type == "efficientnetb0_frozen":
        img = efficientnet_preprocess(img)
    elif backbone_type == "resnet50_frozen":
        img = resnet50_preprocess(img)
    else:
        img = img / 255.0

    return img

def augment_image_np(img):
    if np.random.rand() < 0.5:
        img = np.fliplr(img)
    return img.copy()

# =========================================================
# 8. DATA GENERATOR
# =========================================================
class AblationSequence(tf.keras.utils.Sequence):
    def __init__(
        self,
        df,
        resistance_scaler,
        angular_scaler,
        ba_scaler,
        backbone_type="mobilenetv2_frozen",
        ml_path_column=None,
        use_ml=False,
        use_dic=True,
        use_pr=True,
        batch_size=8,
        shuffle=True,
        training=False
    ):
        self.df = df.reset_index(drop=True)
        self.resistance_scaler = resistance_scaler
        self.angular_scaler = angular_scaler
        self.ba_scaler = ba_scaler
        self.backbone_type = backbone_type
        self.ml_path_column = ml_path_column
        self.use_ml = use_ml
        self.use_dic = use_dic
        self.use_pr = use_pr
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.training = training
        self.indices = np.arange(len(self.df))

        if self.use_ml and self.ml_path_column not in ["ml2d_path", "ml3d_path"]:
            raise ValueError("When use_ml=True, ml_path_column must be 'ml2d_path' or 'ml3d_path'.")

        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_df = self.df.iloc[batch_idx]

        X = {}
        ml_images, dic_images, resistance_vars = [], [], []
        angular_rates, ba_values = [], []

        for _, row in batch_df.iterrows():
            if self.use_ml:
                ml_img = load_and_preprocess_image(
                    row[self.ml_path_column],
                    img_size=IMG_SIZE,
                    backbone_type=self.backbone_type
                )
                if self.training:
                    ml_img = augment_image_np(ml_img)
                ml_images.append(ml_img)

            if self.use_dic:
                dic_img = load_and_preprocess_image(
                    row["dic_path"], img_size=IMG_SIZE, backbone_type=self.backbone_type
                )
                if self.training:
                    dic_img = augment_image_np(dic_img)
                dic_images.append(dic_img)

            if self.use_pr:
                resistance_var = self.resistance_scaler.transform(
                    np.array([[row["resistance_var"]]], dtype=np.float32)
                )[0]
                resistance_vars.append(resistance_var)

            angular_rate = self.angular_scaler.transform(
                np.array([[row["angular_rate"]]], dtype=np.float32)
            )[0]

            ba_value = self.ba_scaler.transform(
                np.array([[row["round_avg_ba"]]], dtype=np.float32)
            )[0]

            angular_rates.append(angular_rate)
            ba_values.append(ba_value)

        if self.use_ml:
            X["ml_input"] = np.array(ml_images, dtype=np.float32)
        if self.use_dic:
            X["dic_input"] = np.array(dic_images, dtype=np.float32)
        if self.use_pr:
            X["res_input"] = np.array(resistance_vars, dtype=np.float32)

        y = {
            "angular_output": np.array(angular_rates, dtype=np.float32),
            "ba_output": np.array(ba_values, dtype=np.float32),
        }

        return X, y

# =========================================================
# 9. BACKBONE BUILDERS
# =========================================================
def make_image_backbone(input_tensor, prefix, backbone_type):
    if backbone_type == "mobilenetv2_frozen":
        backbone = MobileNetV2(
            include_top=False,
            weights="imagenet",
            input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
        )
        backbone._name = f"{prefix}_mobilenetv2"
        backbone.trainable = False
        x = backbone(input_tensor, training=False)

    elif backbone_type == "efficientnetb0_frozen":
        backbone = EfficientNetB0(
            include_top=False,
            weights="imagenet",
            input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
        )
        backbone._name = f"{prefix}_efficientnetb0"
        backbone.trainable = False
        x = backbone(input_tensor, training=False)

    elif backbone_type == "resnet50_frozen":
        backbone = ResNet50(
            include_top=False,
            weights="imagenet",
            input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
        )
        backbone._name = f"{prefix}_resnet50"
        backbone.trainable = False
        x = backbone(input_tensor, training=False)

    else:
        raise ValueError(f"Unknown backbone_type: {backbone_type}")

    x = layers.GlobalAveragePooling2D(name=f"{prefix}_gap")(x)
    x = layers.Dense(256, activation="relu", name=f"{prefix}_dense")(x)
    x = layers.Dropout(0.2, name=f"{prefix}_dropout")(x)
    return x

def make_image_branch(input_name, prefix, backbone_type):
    inp = layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3), name=input_name)
    feat = make_image_backbone(inp, prefix=prefix, backbone_type=backbone_type)
    return inp, feat

def make_pr_branch():
    res_input = layers.Input(shape=(1,), name="res_input")
    x = layers.Dense(64, activation="relu", name="res_dense1")(res_input)
    x = layers.Dense(64, activation="relu", name="res_dense2")(x)
    x = layers.Dense(32, activation="relu", name="res_dense3")(x)
    return res_input, x

def build_model(
    backbone_type="mobilenetv2_frozen",
    use_ml=True,
    use_dic=True,
    use_pr=True,
    learning_rate=LEARNING_RATE
):
    inputs = []
    feature_list = []

    if use_ml:
        ml_input, ml_feat = make_image_branch(
            input_name="ml_input", prefix="ml", backbone_type=backbone_type
        )
        inputs.append(ml_input)
        feature_list.append(ml_feat)

    if use_dic:
        dic_input, dic_feat = make_image_branch(
            input_name="dic_input", prefix="dic", backbone_type=backbone_type
        )
        inputs.append(dic_input)
        feature_list.append(dic_feat)

    if use_pr:
        res_input, res_feat = make_pr_branch()
        inputs.append(res_input)
        feature_list.append(res_feat)

    if len(feature_list) == 0:
        raise ValueError("At least one modality must be selected.")

    if len(feature_list) == 1:
        fused = feature_list[0]
    else:
        fused = layers.Concatenate(name="fusion_concat")(feature_list)

    fused = layers.Dense(256, activation="relu", name="fusion_dense1")(fused)
    fused = layers.Dropout(0.3, name="fusion_dropout1")(fused)
    fused = layers.Dense(128, activation="relu", name="fusion_dense2")(fused)
    fused = layers.Dropout(0.2, name="fusion_dropout2")(fused)

    angular_output = layers.Dense(1, name="angular_output")(fused)
    ba_output = layers.Dense(1, name="ba_output")(fused)

    model = Model(inputs=inputs, outputs=[angular_output, ba_output])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss={
            "angular_output": "mse",
            "ba_output": "mse",
        },
        loss_weights={
            "angular_output": ANGULAR_LOSS_WEIGHT,
            "ba_output": BA_LOSS_WEIGHT,
        }
    )
    return model

# =========================================================
# 10. EVALUATION
# =========================================================
def evaluate_model(model, test_gen, angular_scaler, ba_scaler):
    pred_angular_scaled = []
    pred_ba_scaled = []
    true_angular_scaled = []
    true_ba_scaled = []

    for X_batch, y_batch in test_gen:
        pred_ang_batch, pred_ba_batch = model.predict(X_batch, verbose=0)
        pred_angular_scaled.append(pred_ang_batch.reshape(-1, 1))
        pred_ba_scaled.append(pred_ba_batch.reshape(-1, 1))
        true_angular_scaled.append(y_batch["angular_output"].reshape(-1, 1))
        true_ba_scaled.append(y_batch["ba_output"].reshape(-1, 1))

    pred_angular_scaled = np.vstack(pred_angular_scaled)
    pred_ba_scaled = np.vstack(pred_ba_scaled)
    true_angular_scaled = np.vstack(true_angular_scaled)
    true_ba_scaled = np.vstack(true_ba_scaled)

    pred_angular = angular_scaler.inverse_transform(pred_angular_scaled).reshape(-1)
    pred_ba = ba_scaler.inverse_transform(pred_ba_scaled).reshape(-1)
    true_angular = angular_scaler.inverse_transform(true_angular_scaled).reshape(-1)
    true_ba = ba_scaler.inverse_transform(true_ba_scaled).reshape(-1)

    metrics = {
        "Angular_R2": r2_score(true_angular, pred_angular),
        "Angular_RMSE": np.sqrt(mean_squared_error(true_angular, pred_angular)),
        "BA_R2": r2_score(true_ba, pred_ba),
        "BA_RMSE": np.sqrt(mean_squared_error(true_ba, pred_ba)),
    }

    return metrics, true_angular, pred_angular, true_ba, pred_ba

# =========================================================
# 11. MANUSCRIPT OUTPUTS
# =========================================================
STYLE = {
    "font_family": "Times New Roman",
    "title_size": 16,
    "tick_size": 11,
    "annot_size": 10,
    "colorbar_tick_size": 10,
    "fig_width": 5.8,
    "fig_height": 4.8,
    "bar_fig_width": 15.5,
    "bar_fig_height": 3.0,
    "cell_alpha": 0.85,
    "linecolor": "white",
    "linewidth": 1.2,
    "dpi": 300,
    "cmap_r2": "viridis",
    "cmap_rmse": "viridis_r",
    "fmt": ".2f",
}

plt.rcParams["font.family"] = STYLE["font_family"]

def prepare_results_table(results_df):
    out = results_df.copy()
    out["Modalities"] = pd.Categorical(out["Modalities"], categories=MODALITY_ORDER, ordered=True)
    model_order = [label for _, label in BACKBONE_CONFIGS]
    out["Backbone"] = pd.Categorical(out["Backbone"], categories=model_order, ordered=True)
    return out.sort_values(["Modalities", "Backbone"]).reset_index(drop=True)

def draw_heatmap(data, title, save_path, cmap, metric_type="r2"):
    fig, ax = plt.subplots(figsize=(STYLE["fig_width"], STYLE["fig_height"]))
    fig.patch.set_facecolor("none")
    fig.patch.set_alpha(0)
    ax.set_facecolor("none")

    vals = data.to_numpy(dtype=float)
    finite_vals = vals[np.isfinite(vals)]

    if metric_type == "r2":
        vmin, vmax = 0.0, 1.0
    else:
        vmin, vmax = float(finite_vals.min()), float(finite_vals.max())
        if vmin == vmax:
            vmax = vmin + 1e-9

    im = ax.imshow(vals, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto",
                   alpha=STYLE["cell_alpha"])

    ax.set_xticks(np.arange(data.shape[1]))
    ax.set_yticks(np.arange(data.shape[0]))
    ax.set_xticklabels(data.columns, fontsize=STYLE["tick_size"], rotation=45, ha="right")
    ax.set_yticklabels(data.index, fontsize=STYLE["tick_size"])

    # White cell boundaries
    ax.set_xticks(np.arange(-0.5, data.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, data.shape[0], 1), minor=True)
    ax.grid(which="minor", color=STYLE["linecolor"], linewidth=STYLE["linewidth"])
    ax.tick_params(which="minor", bottom=False, left=False)

    rounded = data.round(2)
    best = rounded.max().max() if metric_type == "r2" else rounded.min().min()

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            value = data.iloc[i, j]
            if pd.isna(value):
                continue
            is_best = round(float(value), 2) == float(best)
            ax.text(
                j, i, format(value, STYLE["fmt"]),
                ha="center", va="center",
                fontsize=STYLE["annot_size"] + (1 if is_best else 0),
                fontweight="bold" if is_best else "normal",
                color="black"
            )

    ax.set_title(title, fontsize=STYLE["title_size"], pad=12, color="black")
    ax.set_xlabel("")
    ax.set_ylabel("")
    for spine in ax.spines.values():
        spine.set_visible(False)

    cbar = fig.colorbar(im, ax=ax)
    cbar.ax.tick_params(labelsize=STYLE["colorbar_tick_size"], colors="black")
    cbar.ax.set_facecolor("none")

    plt.tight_layout()
    plt.savefig(
        save_path, dpi=STYLE["dpi"], bbox_inches="tight",
        transparent=True, facecolor="none", edgecolor="none"
    )
    plt.close(fig)

def save_manuscript_summary(results_df):
    df = prepare_results_table(results_df)
    model_order = [label for _, label in BACKBONE_CONFIGS]

    ba_r2 = df.pivot(index="Modalities", columns="Backbone", values="BA_R2").reindex(MODALITY_ORDER)[model_order]
    ar_r2 = df.pivot(index="Modalities", columns="Backbone", values="Angular_R2").reindex(MODALITY_ORDER)[model_order]
    ba_rmse = df.pivot(index="Modalities", columns="Backbone", values="BA_RMSE").reindex(MODALITY_ORDER)[model_order]
    ar_rmse = df.pivot(index="Modalities", columns="Backbone", values="Angular_RMSE").reindex(MODALITY_ORDER)[model_order]

    draw_heatmap(ba_r2, "BA prediction performance (R²)", os.path.join(FIGURE_DIR, "Figure4b_BA_R2_heatmap.png"), STYLE["cmap_r2"], "r2")
    draw_heatmap(ar_r2, "AR prediction performance (R²)", os.path.join(FIGURE_DIR, "Figure4c_AR_R2_heatmap.png"), STYLE["cmap_r2"], "r2")
    draw_heatmap(ba_rmse, "BA prediction error (RMSE)", os.path.join(FIGURE_DIR, "FigureS23a_BA_RMSE_heatmap.png"), STYLE["cmap_rmse"], "rmse")
    draw_heatmap(ar_rmse, "AR prediction error (RMSE)", os.path.join(FIGURE_DIR, "FigureS23b_AR_RMSE_heatmap.png"), STYLE["cmap_rmse"], "rmse")

    df["Mean_R2"] = df[["BA_R2", "Angular_R2"]].mean(axis=1)
    overall = df.groupby("Modalities", observed=False)["Mean_R2"].mean().reindex(MODALITY_ORDER).reset_index()
    overall = overall.sort_values("Mean_R2", ascending=True).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(STYLE["bar_fig_width"], STYLE["bar_fig_height"]))
    fig.patch.set_facecolor("none")
    fig.patch.set_alpha(0)
    ax.set_facecolor("none")
    ax.patch.set_alpha(0)
    bars = ax.bar(overall["Modalities"].astype(str), overall["Mean_R2"], width=0.72, edgecolor="black", linewidth=1.0)
    best = overall["Mean_R2"].max()
    for bar, val in zip(bars, overall["Mean_R2"]):
        ax.text(bar.get_x()+bar.get_width()/2, val+0.02, f"{val:.2f}", ha="center", va="bottom", fontsize=18, fontweight="bold" if round(val,2)==round(best,2) else "normal", color="black")
    ax.set_ylabel("Overall mean R²", fontsize=18, color="black")
    ax.tick_params(axis="x", labelsize=18, rotation=0, colors="black")
    ax.tick_params(axis="y", labelsize=18, colors="black")
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(0.8)
    ax.grid(False)
    ax.set_ylim(0, overall["Mean_R2"].max()+0.25)
    ax.margins(x=0)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURE_DIR, "Figure4d_Overall_Mean_R2_Barplot.png"), dpi=STYLE["dpi"], bbox_inches="tight", transparent=True, facecolor="none", edgecolor="none")
    plt.close(fig)

    workbook_path = os.path.join(RESULTS_DIR, "results_summary.xlsx")
    with pd.ExcelWriter(workbook_path, engine="openpyxl") as writer:
        df[["Backbone","Modalities","BA_R2","Angular_R2","BA_RMSE","Angular_RMSE"]].to_excel(writer, sheet_name="All_Results", index=False)
        ba_r2.to_excel(writer, sheet_name="Figure4b_BA_R2")
        ar_r2.to_excel(writer, sheet_name="Figure4c_AR_R2")
        overall.to_excel(writer, sheet_name="Figure4d_Overall_R2", index=False)
        ba_rmse.to_excel(writer, sheet_name="FigureS23a_BA_RMSE")
        ar_rmse.to_excel(writer, sheet_name="FigureS23b_AR_RMSE")
    return workbook_path

def save_training_history(history, backbone_label, modality_name):
    history_df = pd.DataFrame(history.history)
    history_df.insert(0, "epoch", np.arange(1, len(history_df) + 1))

    backbone_dir = os.path.join(HISTORY_DIR, safe_name(backbone_label))
    os.makedirs(backbone_dir, exist_ok=True)

    output_path = os.path.join(
        backbone_dir,
        f"{safe_name(modality_name)}_training_history.csv"
    )
    history_df.to_csv(output_path, index=False)
    return output_path


def save_test_predictions(pred_df, backbone_label, modality_name):
    backbone_dir = os.path.join(PREDICTION_DIR, safe_name(backbone_label))
    os.makedirs(backbone_dir, exist_ok=True)

    output_path = os.path.join(
        backbone_dir,
        f"{safe_name(modality_name)}_test_predictions.csv"
    )
    pred_df.to_csv(output_path, index=False)
    return output_path


def save_figure_s24(pred_df):
    freq_labels = {"0p5Hz": "0.5 Hz", "1p0Hz": "1.0 Hz", "2p0Hz": "2.0 Hz"}

    fig, axes = plt.subplots(3, 2, figsize=(11, 11))
    for row, freq in enumerate(FREQUENCY_FOLDERS):
        sub = pred_df[pred_df["frequency_folder"] == freq].copy().sort_values("time_s")
        x = np.arange(len(sub))

        ax = axes[row, 0]
        ax.plot(x, sub["true_ba"], color="black", linewidth=1.5, label="True")
        ax.scatter(x, sub["pred_ba"], color="red", s=35, label="Predicted", zorder=3)
        ax.text(0.02, 0.94, freq_labels[freq], transform=ax.transAxes,
                ha="left", va="top", fontsize=15)
        ax.legend(loc="upper right", frameon=False, fontsize=11)

        ax = axes[row, 1]
        ax.plot(x, sub["true_angular"], color="black", linewidth=1.5, label="True")
        ax.scatter(x, sub["pred_angular"], color="red", s=35, label="Predicted", zorder=3)
        ax.text(0.02, 0.94, freq_labels[freq], transform=ax.transAxes,
                ha="left", va="top", fontsize=15)
        ax.legend(loc="upper right", frameon=False, fontsize=11)

    for ax in axes.flat:
        ax.tick_params(labelsize=10)

    plt.tight_layout()
    plt.savefig(
        os.path.join(FIGURE_DIR, "FigureS24_True_vs_Predicted.png"),
        dpi=STYLE["dpi"], bbox_inches="tight"
    )
    plt.close(fig)

# =========================================================
# 12. TRAINING AND TESTING
# =========================================================
def main():
    full_df = build_full_dataframe(DATA_DIR, FREQUENCY_FOLDERS)
    train_df = full_df[full_df["cycle_num"].isin(TRAIN_CYCLES)].reset_index(drop=True)
    test_df = full_df[full_df["cycle_num"].isin(TEST_CYCLES)].reset_index(drop=True)

    print(f"Total samples: {len(full_df)}")
    print(f"Training samples: {len(train_df)} | cycles {TRAIN_CYCLES}")
    print(f"Testing samples: {len(test_df)} | cycles {TEST_CYCLES}")

    resistance_scaler = StandardScaler().fit(train_df[["resistance_var"]].values.astype(np.float32))
    angular_scaler = StandardScaler().fit(train_df[["angular_rate"]].values.astype(np.float32))
    ba_scaler = StandardScaler().fit(train_df[["round_avg_ba"]].values.astype(np.float32))

    all_results = []
    figure_s24_predictions = None

    for backbone_key, backbone_label in BACKBONE_CONFIGS:
        for cfg in EXPERIMENT_CONFIGS:
            modality_name = cfg["Modalities"]
            exp_name = experiment_name(backbone_key, modality_name)

            print("\n" + "=" * 90)
            print(f"Backbone: {backbone_label} | Modalities: {modality_name}")
            print("=" * 90)

            tf.keras.backend.clear_session()
            set_all_seeds(SEED)

            train_gen = AblationSequence(
                train_df, resistance_scaler, angular_scaler, ba_scaler,
                backbone_type=backbone_key, ml_path_column=cfg["ML_Path_Column"],
                use_ml=cfg["Use_ML"], use_dic=cfg["Use_DIC"], use_pr=cfg["Use_PR"],
                batch_size=BATCH_SIZE, shuffle=True, training=True
            )
            test_gen = AblationSequence(
                test_df, resistance_scaler, angular_scaler, ba_scaler,
                backbone_type=backbone_key, ml_path_column=cfg["ML_Path_Column"],
                use_ml=cfg["Use_ML"], use_dic=cfg["Use_DIC"], use_pr=cfg["Use_PR"],
                batch_size=BATCH_SIZE, shuffle=False, training=False
            )

            model = build_model(
                backbone_type=backbone_key,
                use_ml=cfg["Use_ML"], use_dic=cfg["Use_DIC"], use_pr=cfg["Use_PR"],
                learning_rate=LEARNING_RATE
            )

            weights_path = os.path.join(MODEL_DIR, f"{exp_name}.weights.h5")
            callbacks = [
                tf.keras.callbacks.ModelCheckpoint(
                    filepath=weights_path, monitor="loss", save_best_only=True,
                    save_weights_only=True, verbose=0
                ),
                tf.keras.callbacks.ReduceLROnPlateau(
                    monitor="loss", factor=0.5, patience=3, min_lr=1e-6, verbose=0
                ),
            ]

            history = model.fit(
                train_gen,
                epochs=EPOCHS,
                callbacks=callbacks,
                verbose=1
            )

            save_training_history(
                history,
                backbone_label=backbone_label,
                modality_name=modality_name
            )

            if os.path.exists(weights_path):
                model.load_weights(weights_path)

            metrics, true_ang, pred_ang, true_ba, pred_ba = evaluate_model(
                model, test_gen, angular_scaler, ba_scaler
            )

            row = {
                "Backbone": backbone_label,
                "Modalities": modality_name,
                **metrics,
            }
            all_results.append(row)
            print({k: round(v, 4) for k, v in metrics.items()})

            prediction_df = pd.DataFrame({
                "frequency_folder": test_df["frequency_folder"].values,
                "cycle_num": test_df["cycle_num"].values,
                "time_s": test_df["time_s"].values,
                "true_ba": true_ba,
                "pred_ba": pred_ba,
                "true_angular": true_ang,
                "pred_angular": pred_ang,
            })

            save_test_predictions(
                prediction_df,
                backbone_label=backbone_label,
                modality_name=modality_name
            )

            if backbone_label == "ResNet50" and modality_name == "3D ML + DIC + PR":
                figure_s24_predictions = prediction_df.copy()

    results_df = pd.DataFrame(all_results)
    results_df = prepare_results_table(results_df)
    results_df.to_csv(os.path.join(RESULTS_DIR, "all_experiment_results.csv"), index=False)
    results_df.to_excel(os.path.join(RESULTS_DIR, "all_experiment_results.xlsx"), index=False)

    summary_path = save_manuscript_summary(results_df)
    if figure_s24_predictions is None:
        raise RuntimeError("Figure S24 prediction data were not generated.")
    save_figure_s24(figure_s24_predictions)

    print("\nCompleted.")
    print("Results:", RESULTS_DIR)
    print("Summary:", summary_path)
    print("Figures:", FIGURE_DIR)
    print("Test predictions:", PREDICTION_DIR)
    print("Training history:", HISTORY_DIR)

if __name__ == "__main__":
    main()
